Author: Aparajita Nath

Product: split the data and set up the preprocessing, with no information leaking from the test set

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import helpers as h

pd.set_option("display.max_columns", 30)

## 2. Data preprocessing

### 2.1 Load the cleaned data (from A)

In [2]:
df = pd.read_csv(h.PROCESSED / "credit_clean.csv")
X = df.drop(columns="bad_credit")
y = df["bad_credit"]
print(X.shape, f"bad share: {y.mean():.1%}")

(1000, 20) bad share: 30.0%


### 2.2 Divide the data into a training set and a held-out test set

70% training and 30% test, stratified on the outcome so that both parts keep the same bad share. The test set is used only to report performance. Hyperparameters are tuned by cross-validation inside the training set, so the test set never influences a modelling choice.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=h.SEED)
print(f"Train: {len(X_train)} cases ({y_train.mean():.1%} bad)")
print(f"Test:  {len(X_test)} cases ({y_test.mean():.1%} bad)")

X_train.to_csv(h.PROCESSED / "X_train.csv")
X_test.to_csv(h.PROCESSED / "X_test.csv")
y_train.to_csv(h.PROCESSED / "y_train.csv")
y_test.to_csv(h.PROCESSED / "y_test.csv")

Train: 700 cases (30.0% bad)
Test:  300 cases (30.0% bad)


### 2.3 Preprocessing

- Numeric variables (`duration`, `amount`, `age`): fill gaps with the median and standardise (there are no gaps in this dataset, so the step is a safeguard).
- Categorical variables: fill gaps with the most frequent value and one-hot encode.

The preprocessing is not applied to the data here and saved. It is part of each model's pipeline, so it is fitted on the training data only, and again on the training part of each cross-validation fold. Scaling or encoding the whole dataset before splitting would let information from the test set leak into the model.

In [4]:
categorical = h.categorical_columns(X_train)
preprocess = h.make_preprocess(categorical)

train_matrix = preprocess.fit_transform(X_train)          # fitted on the training set only
test_matrix = preprocess.transform(X_test)                # test set only transformed
names = preprocess.get_feature_names_out()
print(f"Input columns: {X_train.shape[1]}  ->  model features after encoding: {len(names)}")

def first_three_means(matrix):
    return np.asarray(matrix[:, :3].mean(axis=0)).ravel()      # works for dense and sparse output

check = pd.DataFrame({"train mean": first_three_means(train_matrix),
                      "test mean": first_three_means(test_matrix)}, index=h.CONTINUOUS)
print("\nStandardised numeric columns: train mean is 0 by construction, test mean is not")
check.round(3)

Input columns: 20  ->  model features after encoding: 71

Standardised numeric columns: train mean is 0 by construction, test mean is not


,train mean,test mean
duration,0.0,0.039
amount,0.0,0.070
age,-0.0,-0.075


### 2.4 Class imbalance

The sample has 30% bad credits. The data are not resampled (no under- or over-sampling), because the aim is to compare predicted probabilities with observed rates, and resampling would change the bad share the models learn from. Ranking is judged with ROC-AUC, which does not depend on the bad share, and probability accuracy with the Brier score and calibration.

### 2.5 Correlation between the numeric variables (training set)

In [5]:
X_train[h.CONTINUOUS].corr().round(2)

,duration,amount,age
duration,1.00,0.58,-0.03
amount,0.58,1.00,0.03
age,-0.03,0.03,1.00


`duration` and `amount` are positively correlated (longer loans are usually larger). Neither is correlated with `age` to a degree that would cause a problem for a logistic regression.